In [ ]:
import sys
!{sys.executable} -m pip install "setuptools<70.0.0"

In [ ]:
import pkg_resources
import mlflow
print("✅ Dependency Hell Defeated!")

In [ ]:
import os
import mlflow
import optuna
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import xgboost as xgb
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
from pyspark.sql.functions import unix_timestamp
from pyiceberg.catalog import load_catalog 
from impala.dbapi import connect
from impala.util import as_pandas


def get_chronological_splits(df, timestamp_col='collection_timestamp'):
    """
    Strict chronological split per Project Kukui PRD.
    Train: Months 1-18 | Val: Months 19-21 | Test: Months 22-24
    """
    print("Sorting by cell and time to prevent data leakage...")
    df = df.sort_values(by=['cell_id', timestamp_col])
    
    # Drop the first 672 rows per cell (1 week) because t-672 lags will be NULL
    print("Dropping initial 672 intervals per cell (lag initialization)...")
    df = df.groupby('cell_id').apply(lambda x: x.iloc[672:]).reset_index(drop=True)
    
    # Determine the date cutoffs based on the 24-month synthetic data window
    # Assuming start date is 2023-01-01
    train_end = pd.to_datetime('2024-06-30')
    val_end   = pd.to_datetime('2024-09-30')
    
    df[timestamp_col] = pd.to_datetime(df[timestamp_col])
    
    train = df[df[timestamp_col] <= train_end]
    val   = df[(df[timestamp_col] > train_end) & (df[timestamp_col] <= val_end)]
    test  = df[df[timestamp_col] > val_end]
    
    print(f"Train size: {len(train)} | Val size: {len(val)} | Test size: {len(test)}")
    return train, val, test


# 1. Load Data via CDW (Bypassing Spark and PyIceberg entirely)
print("Connecting to CDW Impala Virtual Warehouse...")

IMPALA_HOST = 'coordinator-ps-amer-impala-warehouse.dw-ps-amer-sandbox-aws.a465-9q4k.cloudera.site' 
WORKLOAD_USER = 'athangamani'
WORKLOAD_PASSWORD = 'BlancaLake123'

conn = connect(
    host=IMPALA_HOST, port=443, use_ssl=True, auth_mechanism='LDAP',
    user=WORKLOAD_USER, password=WORKLOAD_PASSWORD, use_http_transport=True, http_path='cliservice'
)
cursor = conn.cursor()

# THE FIX: Dynamically find a valid cell tower in Chicago
print("Finding a valid Cell ID in Chicago...")
cursor.execute("SELECT cell_id FROM ohana.ml_feature_store WHERE market = 'Chicago' LIMIT 1")
valid_cell_id = cursor.fetchone()[0]
print(f"Targeting Cell: {valid_cell_id}")

print("Executing C++ MPP engine scan on Iceberg Data Lake...")
query = f"""
    SELECT * FROM ohana.ml_feature_store 
    WHERE market = 'Chicago' 
      AND confidence_tier = 'HIGH'
      AND cell_id = '{valid_cell_id}'
"""
cursor.execute(query)

print("Transferring results directly into Pandas...")
df_pandas = as_pandas(cursor)
print(f"✅ Successfully loaded {len(df_pandas):,} rows from CDW into memory!")

# Define Features and Target (Drop non-ML columns)
TARGET = 'dl_prb_utilization_pct'
FEATURES = [c for c in df_pandas.columns if c not in [
    'cell_id', 'enb_id', 'market', 'collection_date', 
    'collection_timestamp', 'is_outage', 'is_saturated', 'dq_passed', 
    'load_ts', 'confidence_tier', TARGET
]]

# Force Impala objects into native floats for XGBoost
print("Casting Impala objects to native floats for XGBoost...")
for col in FEATURES:
    df_pandas[col] = pd.to_numeric(df_pandas[col], errors='coerce')
df_pandas[TARGET] = pd.to_numeric(df_pandas[TARGET], errors='coerce')

# 2. Split Data Chronologically
train, val, test = get_chronological_splits(df_pandas)

X_train, y_train = train[FEATURES], train[TARGET]
X_val, y_val = val[FEATURES], val[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

# 3. Optuna Objective for Hyperparameter Tuning
def objective(trial):
    params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'mape',
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'random_state': 42
    }
    
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    
    preds = model.predict(X_val)
    # MAPE calculation
    val_mape = mean_absolute_percentage_error(y_val, preds) * 100
    return val_mape

# 4. MLflow Tracking & Optuna Search
mlflow.set_experiment("ohana-traffic-prediction-1hr")

with mlflow.start_run(run_name="XGBoost_Optuna_Champion"):
    print("Starting Optuna Hyperparameter Search (10 Trials)...")
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=10) # Set to 50 for final production run
    
    best_params = study.best_params
    print(f"Best Validation MAPE: {study.best_value:.2f}%")
    
    # 5. Train the Final Champion Model
    print("Training final Champion Model with best parameters...")
    champion_model = xgb.XGBRegressor(**best_params, objective='reg:squarederror')
    champion_model.fit(X_train, y_train)
    
    # 6. Evaluate on the HELD-OUT TEST SET
    test_preds = champion_model.predict(X_test)
    test_mape = mean_absolute_percentage_error(y_test, test_preds) * 100
    test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
    
    print(f"🏆 Final Held-Out Test MAPE: {test_mape:.2f}%")
    
    # 7. Log Everything to MLflow
    mlflow.log_params(best_params)
    mlflow.log_metrics({"val_mape": study.best_value, "test_mape": test_mape, "test_rmse": test_rmse})
    mlflow.set_tags({
        "model_type": "xgboost", 
        "forecast_horizon": "1h", 
        "target_variable": TARGET
    })
    
    # Log the actual model artifact to the Tracking Server
    mlflow.xgboost.log_model(champion_model, "xgb_model")
    print("✅ Model successfully tracked in MLflow!")

    # ---------------------------------------------------------
    # 8. DIRECT JSON EXPORT (The Bypass Strategy)
    # ---------------------------------------------------------
    print("Exporting native JSON model for direct API deployment...")
    
    # Ensure the target directory exists
    os.makedirs("/home/cdsw/models", exist_ok=True)
    
    # Save the pure XGBoost model natively
    champion_model.save_model("/home/cdsw/models/ohana_wireless_model.json")
    
    print("✅ Native model saved to /home/cdsw/models/ohana_wireless_model.json")